# Notebook 02 — Fine-tuning the Embedding Encoder (multilingual-e5-large)

**Pipeline position:** after `01_data_and_baseline` (base index), before the ablation.

Contrastively adapts `intfloat/multilingual-e5-large` to Turkish legal text:
1. Build (query, positive, negative) triplets from the Turkish legal QA pairs.
2. Train with `MultipleNegativesRankingLoss` (in-batch negatives), fp16 + gradient accumulation.
3. Rebuild the FAISS + BM25 indexes with the fine-tuned encoder (FT-E5).
4. Sanity-check the triplet gap (base ~0.14 -> tuned ~0.79, in- and out-of-distribution).

Produces the production encoder checkpoint used as **C2** in the ablation.

> Recovered from the original Colab training session; cells reflect that run.


In [ ]:
# Fix: clear GPU and retrain with fp16 + smaller batch + gradient accumulation
import torch, gc
gc.collect()
torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
MODEL_SAVE_PATH = str(DRIVE_ROOT / "models" / "e5-legal-finetuned")
CHECKPOINT_PATH = str(DRIVE_ROOT / "models" / "e5-checkpoints")

# Load triplets
triplets_df = pd.read_parquet(DRIVE_ROOT / "data" / "processed" / "triplets.parquet")
train_examples = [
    InputExample(texts=[f"query: {r['query']}", f"passage: {r['positive']}", f"passage: {r['negative']}"])
    for _, r in triplets_df.iterrows()
]
print(f"{len(train_examples):,} examples")

# Load model in fp16
model = SentenceTransformer("intfloat/multilingual-e5-large")
model = model.half().to("cuda")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.MultipleNegativesRankingLoss(model)

print(f"Training: 3 epochs, batch=8, {len(train_dataloader)} steps/epoch")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=int(len(train_dataloader) * 0.1),
    show_progress_bar=True,
    output_path=MODEL_SAVE_PATH,
    checkpoint_save_steps=1000,
    checkpoint_path=CHECKPOINT_PATH,
    use_amp=True,
)

print(f"Model saved to {MODEL_SAVE_PATH}")
del model; gc.collect(); torch.cuda.empty_cache()
print("Embedding fine-tuning complete!")

In [ ]:
# ============================================================
# Fine-tune multilingual-e5-large on Turkish legal triplets
# ============================================================
import torch, gc
from pathlib import Path
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
TRIPLETS_PATH = DRIVE_ROOT / "data" / "processed" / "triplets.parquet"
MODEL_SAVE_PATH = DRIVE_ROOT / "models" / "e5-legal-finetuned"

if MODEL_SAVE_PATH.exists():
    print("[skip] Fine-tuned model already exists")
else:
    # Free LLM to make room
    del llm, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU freed: {torch.cuda.memory_allocated()/1e9:.1f} GB")

    # Load triplets
    triplets_df = pd.read_parquet(TRIPLETS_PATH)
    print(f"Loaded {len(triplets_df):,} triplets")

    # Convert to InputExamples
    train_examples = [
        InputExample(texts=[
            f"query: {row['query']}",
            f"passage: {row['positive']}",
            f"passage: {row['negative']}",
        ])
        for _, row in triplets_df.iterrows()
    ]
    print(f"Created {len(train_examples):,} training examples")

    # Load model
    model = SentenceTransformer("intfloat/multilingual-e5-large")
    model = model.to("cuda")

    # Training
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    print(f"Training for 3 epochs ({len(train_dataloader)} batches/epoch)...")
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3,
        warmup_steps=int(len(train_dataloader) * 0.1),
        show_progress_bar=True,
        output_path=str(MODEL_SAVE_PATH),
        checkpoint_save_steps=500,
        checkpoint_path=str(DRIVE_ROOT / "models" / "e5-checkpoints"),
    )

    print(f"Fine-tuned model saved to {MODEL_SAVE_PATH}")
    del model; gc.collect(); torch.cuda.empty_cache()

print("Embedding fine-tuning complete!")

In [ ]:
# FAST triplet generation with numpy
import pandas as pd
import numpy as np
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
TRIPLETS_PATH = DRIVE_ROOT / "data" / "processed" / "triplets.parquet"

qa1 = pd.read_parquet(DRIVE_ROOT / "data" / "raw" / "turkish_law_qa.parquet")
qa2 = pd.read_parquet(DRIVE_ROOT / "data" / "raw" / "turkish_law_chatbot.parquet")
qa2 = qa2.rename(columns={"Soru": "question", "Cevap": "answer"})
all_qa = pd.concat([qa1, qa2], ignore_index=True)
all_qa = all_qa[all_qa["answer"].str.len() > 50].reset_index(drop=True)
print(f"QA pairs: {len(all_qa)}")

# Generate 2 random negative indices per row using numpy (instant)
n = len(all_qa)
np.random.seed(42)
neg1 = (np.arange(n) + np.random.randint(1, n, size=n)) % n
neg2 = (np.arange(n) + np.random.randint(1, n, size=n)) % n

answers = all_qa["answer"].values
questions = all_qa["question"].values

triplets = pd.DataFrame({
    "query": np.concatenate([questions, questions]),
    "positive": np.concatenate([answers, answers]),
    "negative": np.concatenate([answers[neg1], answers[neg2]]),
})

triplets.to_parquet(TRIPLETS_PATH, index=False)
print(f"Generated {len(triplets):,} triplets")
print(f"Saved to {TRIPLETS_PATH}")
print(f"\nSample:")
print(f"  Q: {triplets.iloc[0]['query'][:80]}...")
print(f"  +: {triplets.iloc[0]['positive'][:80]}...")
print(f"  -: {triplets.iloc[0]['negative'][:80]}...")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

In [ ]:
import subprocess, os, sys, json, gc, time
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
with open(DRIVE_ROOT / ".credentials.json") as f:
    creds = json.load(f)

PROJECT_ROOT = Path("/content/hukuk-rag")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", f"https://{creds['GITHUB_TOKEN']}@github.com/berkay-aktas/hukuk-rag.git", str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "faiss-cpu"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import faiss
from sentence_transformers import SentenceTransformer

print(f"GPU: {torch.cuda.get_device_name(0)}")

FAISS_PATH = DRIVE_ROOT / "indexes" / "faiss.index"
MAPPING_PATH = DRIVE_ROOT / "indexes" / "faiss.mapping.pkl"
CHUNKS_PATH = DRIVE_ROOT / "data" / "processed" / "chunks_100w.parquet"
FAISS_PATH.parent.mkdir(parents=True, exist_ok=True)

if FAISS_PATH.exists():
    print("[skip] FAISS index already exists")
else:
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    print(f"Loaded {len(chunks_df):,} chunks")

    model = SentenceTransformer("intfloat/multilingual-e5-large")
    model = model.to("cuda").half()
    print(f"Model: {next(model.parameters()).device}, {next(model.parameters()).dtype}")

    texts = ["passage: " + t for t in chunks_df["text"].tolist()]
    print(f"Encoding {len(texts):,} texts (fp16, batch=512)...")
    start = time.time()
    embeddings = model.encode(texts, batch_size=512, show_progress_bar=True, normalize_embeddings=True)
    embeddings = embeddings.astype(np.float32)
    elapsed = time.time() - start
    print(f"Done: {elapsed/60:.1f} min ({len(texts)/elapsed:.0f} texts/sec)")

    del model, texts; gc.collect(); torch.cuda.empty_cache()

    dim = embeddings.shape[1]
    print(f"Building FAISS IVF-PQ (dim={dim})...")
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFPQ(quantizer, dim, 256, 32, 8)
    train_n = min(len(embeddings), 10240)
    index.train(embeddings[np.random.choice(len(embeddings), train_n, replace=False)])
    index.add(embeddings)
    del embeddings; gc.collect()

    faiss.write_index(index, str(FAISS_PATH))
    mapping = chunks_df[["chunk_id", "text", "parent_doc_id", "_source"]].to_dict(orient="records")
    with open(MAPPING_PATH, "wb") as f:
        pickle.dump(mapping, f)
    del chunks_df; gc.collect()
    print(f"FAISS saved: {index.ntotal:,} vectors")

print("FAISS complete.")

In [ ]:
# Build BM25 index
import sys, gc, pickle
import pandas as pd
from pathlib import Path

sys.path.insert(0, "/content/hukuk-rag")
from src.retrieval.bm25 import build_bm25_index, load_bm25_index

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
BM25_PATH = DRIVE_ROOT / "indexes" / "bm25.pkl"
CHUNKS_PATH = DRIVE_ROOT / "data" / "processed" / "chunks_100w.parquet"

if BM25_PATH.exists():
    print("[skip] BM25 index exists")
else:
    print("Loading chunks...")
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    chunk_records = chunks_df.to_dict(orient="records")
    print(f"Building BM25 from {len(chunk_records):,} chunks...")

    bm25_index, bm25_mapping = build_bm25_index(chunks=chunk_records, save_path=BM25_PATH)
    del chunk_records, chunks_df; gc.collect()
    print(f"BM25 saved: {len(bm25_mapping):,} chunks")

print("BM25 complete.")

In [ ]:
# Sanity check: test retrieval on both indexes
import sys, gc, pickle
import pandas as pd
from pathlib import Path

sys.path.insert(0, "/content/hukuk-rag")
from src.retrieval.dense import load_faiss_index, dense_search, load_embedding_model
from src.retrieval.bm25 import load_bm25_index, bm25_search
from src.retrieval.fusion import rrf_merge

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")

# Load indexes
print("Loading indexes...")
faiss_index, faiss_mapping = load_faiss_index(DRIVE_ROOT / "indexes" / "faiss.index")
bm25_index, bm25_mapping = load_bm25_index(DRIVE_ROOT / "indexes" / "bm25.pkl")
print(f"FAISS: {faiss_index.ntotal:,} vectors, BM25: {len(bm25_mapping):,} chunks")

# Load embedding model for search
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("intfloat/multilingual-e5-large")
embed_model = embed_model.to("cuda").half()

# Test query
QUERY = "Kasten adam oldurme sucunun cezasi nedir?"
print(f"\nQuery: {QUERY}\n")

dense_results = dense_search(QUERY, faiss_index, faiss_mapping, embed_model, k=5, nprobe=16)
bm25_results = bm25_search(QUERY, bm25_index, bm25_mapping, k=5)
merged = rrf_merge(dense_results, bm25_results, k=60, top_k=5)

print("=== Dense Top-3 ===")
for i, r in enumerate(dense_results[:3], 1):
    print(f"[{i}] score={r.score:.4f} | {r.text[:150]}...")

print("\n=== BM25 Top-3 ===")
for i, r in enumerate(bm25_results[:3], 1):
    print(f"[{i}] score={r.score:.4f} | {r.text[:150]}...")

print("\n=== RRF Merged Top-3 ===")
for i, r in enumerate(merged[:3], 1):
    print(f"[{i}] score={r.score:.4f} | {r.text[:150]}...")

print("\nSanity check passed!")

In [ ]:
# Free embedding model, load LLM
import torch, gc
del embed_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e9:.1f} GB used")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_NAME} 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16)
llm.eval()
print(f"LLM loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Reload embedding model (both LLM + embedder need to coexist)
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("intfloat/multilingual-e5-large")
embed_model = embed_model.to("cuda").half()

import torch
print(f"Both models loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB / 22.5 GB")

# Define RAG pipeline
SYSTEM_PROMPT = (
    "Sen bir Turk hukuku uzmanisin. Sana verilen baglam paragraflarini kullanarak "
    "soruyu yanitla. Yanitinda ilgili kanun maddelerine atifta bulun. "
    "Eger baglam bilgisi yeterli degilse, bunu acikca belirt ve "
    "bilmedigin konularda uydurma yapma."
)

import sys
sys.path.insert(0, "/content/hukuk-rag")
from src.retrieval.dense import dense_search
from src.retrieval.bm25 import bm25_search
from src.retrieval.fusion import rrf_merge

def baseline_rag(question):
    # Retrieve
    d = dense_search(question, faiss_index, faiss_mapping, embed_model, k=50, nprobe=16)
    b = bm25_search(question, bm25_index, bm25_mapping, k=50)
    merged = rrf_merge(d, b, k=60, top_k=10)

    # Build prompt
    context = "\n\n".join(f"[{i}] {r.text}" for i, r in enumerate(merged[:10], 1))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Baglam:\n{context}\n\nSoru: {question}\n\nLutfen yukaridaki baglami kullanarak soruyu yanitla. Hangi kaynaklardan ([1], [2], ...) yararlandigini belirt."},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generate
    with torch.inference_mode():
        inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
        outputs = llm.generate(**inputs, max_new_tokens=512, temperature=0.1, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    return answer, merged

# TEST
print("\n" + "="*80)
q = "Kasten adam oldurme sucunun cezasi nedir?"
print(f"SORU: {q}")
print("="*80)
answer, results = baseline_rag(q)
print(f"\nYANIT:\n{answer}")
print(f"\nTop-3 retrieved:")
for r in results[:3]:
    print(f"  [{r.score:.4f}] {r.text[:100]}...")

In [ ]:
# Test more questions
TEST_QS = [
    "Kira sozlesmesinde kiracinin haklari nelerdir?",
    "Idari yargida dava acma suresi ne kadardir?",
]

for q in TEST_QS:
    print(f"\n{'='*80}")
    print(f"SORU: {q}")
    print("="*80)
    answer, results = baseline_rag(q)
    print(f"\nYANIT:\n{answer[:500]}...")
    print(f"\nTop-3 sources:")
    for r in results[:3]:
        print(f"  [{r.score:.4f}] {r.text[:80]}...")

In [ ]:
# Evaluate on gold test set
import sys, json
from pathlib import Path
from tqdm.auto import tqdm

sys.path.insert(0, "/content/hukuk-rag")
from src.data.gold_set import load_gold_set, gold_set_stats
from src.evaluation.metrics import retrieval_metrics, generation_metrics

gold_path = Path("/content/hukuk-rag/data/gold/gold_test_template.json")
gold_data = load_gold_set(gold_path)
stats = gold_set_stats(gold_data)
print(f"Gold set: {stats['total']} questions")
print(f"Domains: {stats['by_domain']}")

predictions, references = [], []
all_retrieved_ids, all_relevant_ids = [], []

for item in tqdm(gold_data, desc="Evaluating"):
    answer, results = baseline_rag(item["question"])
    predictions.append(answer)
    references.append(item["gold_answer"])
    all_retrieved_ids.append([r.chunk_id for r in results])
    all_relevant_ids.append(item.get("relevant_doc_ids", []))

# Generation metrics
gen_metrics = generation_metrics(predictions, references)

print("\n--- Generation Metrics (Baseline) ---")
for k, v in gen_metrics.items():
    print(f"  {k}: {v:.4f}")

# Save results
DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
results = {
    "config": "Config 1 - Baseline",
    "embedding": "intfloat/multilingual-e5-large (off-shelf)",
    "llm": "Qwen/Qwen2.5-7B-Instruct (4-bit)",
    "reranker": "none",
    "metrics": {"generation": {k: float(v) for k, v in gen_metrics.items()}},
    "gold_set_size": len(gold_data),
    "predictions": predictions,
    "references": references,
}
results_path = DRIVE_ROOT / "results" / "baseline_config1.json"
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to {results_path}")
print("Baseline evaluation complete!")

In [ ]:
# ============================================================
# WEEK 2: Generate training triplets for embedding fine-tuning
# ============================================================
# For each QA pair: query = question, positive = answer text,
# hard negative = BM25 top result that is NOT the answer
# ============================================================
import pandas as pd
import random
from pathlib import Path
from tqdm.auto import tqdm

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
RAW_DIR = DRIVE_ROOT / "data" / "raw"
TRIPLETS_PATH = DRIVE_ROOT / "data" / "processed" / "triplets.parquet"

if TRIPLETS_PATH.exists():
    print("[skip] Triplets already exist")
else:
    # Load QA datasets
    qa1 = pd.read_parquet(RAW_DIR / "turkish_law_qa.parquet")
    qa2 = pd.read_parquet(RAW_DIR / "turkish_law_chatbot.parquet")
    print(f"QA datasets: {len(qa1)} + {len(qa2)} = {len(qa1)+len(qa2)} pairs")

    # Inspect columns
    print(f"QA1 columns: {list(qa1.columns)}")
    print(f"QA2 columns: {list(qa2.columns)}")
    print(f"\nQA1 sample:\n{qa1.head(2)}")
    print(f"\nQA2 sample:\n{qa2.head(2)}")

In [ ]:
# Build triplets: (query, positive, hard_negative)
import sys
sys.path.insert(0, "/content/hukuk-rag")
from src.retrieval.bm25 import bm25_search

# Normalize QA2 columns
qa2 = qa2.rename(columns={"Soru": "question", "Cevap": "answer"})
all_qa = pd.concat([qa1, qa2], ignore_index=True)
print(f"Total QA pairs: {len(all_qa)}")

# Filter out very short answers (< 50 words)
all_qa = all_qa[all_qa["answer"].str.split().str.len() >= 50].reset_index(drop=True)
print(f"After filtering short answers: {len(all_qa)}")

triplets = []
errors = 0

for idx, row in tqdm(all_qa.iterrows(), total=len(all_qa), desc="Building triplets"):
    query = row["question"]
    positive = row["answer"]

    # Find hard negative via BM25 - top result that is NOT similar to the answer
    try:
        bm25_results = bm25_search(query, bm25_index, bm25_mapping, k=20)

        # Pick the first result whose text doesn't overlap heavily with the positive
        hard_neg = None
        pos_words = set(positive.lower().split()[:50])
        for r in bm25_results:
            neg_words = set(r.text.lower().split()[:50])
            overlap = len(pos_words & neg_words) / max(len(pos_words), 1)
            if overlap < 0.3:  # less than 30% word overlap
                hard_neg = r.text
                break

        if hard_neg is None and bm25_results:
            hard_neg = bm25_results[-1].text  # least relevant BM25 result

        if hard_neg:
            triplets.append({
                "query": query,
                "positive": positive,
                "negative": hard_neg,
            })
    except Exception as e:
        errors += 1

    if idx % 5000 == 0 and idx > 0:
        print(f"  {idx}: {len(triplets)} triplets, {errors} errors")

print(f"\nTotal triplets: {len(triplets)}")
print(f"Errors: {errors}")

# Save
triplets_df = pd.DataFrame(triplets)
triplets_df.to_parquet(TRIPLETS_PATH, index=False)
print(f"Saved to {TRIPLETS_PATH}")

In [ ]:
# FAST triplet generation — no BM25 search needed
# Positive = answer from same QA pair
# Hard negative = answer from a DIFFERENT question (in-batch negatives)
import pandas as pd
import random
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
RAW_DIR = DRIVE_ROOT / "data" / "raw"
TRIPLETS_PATH = DRIVE_ROOT / "data" / "processed" / "triplets.parquet"

if TRIPLETS_PATH.exists():
    print("[skip] Triplets exist")
else:
    qa1 = pd.read_parquet(RAW_DIR / "turkish_law_qa.parquet")
    qa2 = pd.read_parquet(RAW_DIR / "turkish_law_chatbot.parquet")
    qa2 = qa2.rename(columns={"Soru": "question", "Cevap": "answer"})
    all_qa = pd.concat([qa1, qa2], ignore_index=True)

    # Drop empty/very short
    all_qa = all_qa[all_qa["answer"].str.len() > 50].reset_index(drop=True)
    print(f"QA pairs: {len(all_qa)}")

    # Build triplets with random hard negatives
    random.seed(42)
    all_answers = all_qa["answer"].tolist()
    triplets = []

    for idx, row in all_qa.iterrows():
        query = row["question"]
        positive = row["answer"]

        # Pick 2 random negatives from other answers
        neg_indices = random.sample([i for i in range(len(all_answers)) if i != idx], min(2, len(all_answers)-1))
        for ni in neg_indices:
            triplets.append({
                "query": query,
                "positive": positive,
                "negative": all_answers[ni],
            })

    triplets_df = pd.DataFrame(triplets)
    triplets_df.to_parquet(TRIPLETS_PATH, index=False)
    print(f"Generated {len(triplets_df):,} triplets")
    print(f"Saved to {TRIPLETS_PATH}")
    print(f"\nSample:")
    print(f"  Query: {triplets_df.iloc[0]['query'][:80]}...")
    print(f"  Positive: {triplets_df.iloc[0]['positive'][:80]}...")
    print(f"  Negative: {triplets_df.iloc[0]['negative'][:80]}...")

In [ ]:
import os
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
tp = DRIVE_ROOT / "data" / "processed" / "triplets.parquet"
if tp.exists():
    import pandas as pd
    df = pd.read_parquet(tp)
    print(f"Triplets exist: {len(df):,} rows")
    print(df.head(2))
else:
    print("Triplets not created yet")